# 4.1 Generate historical predictions (2022–2025)

Run after `ML/6_save_final_model.ipynb` and both notebooks in `Heuristic/`. Loads the saved models and produces actual prices plus ML, value-signal, naive, operating-protocol, and AEMO forecasts for calendar years 2022–2025. Every origin at `BACKTEST_ORIGIN_CADENCE` retains all 96 half-hour horizons. Processes one month at a time and writes the combined CSV to `actual_vs_predicted_test_set.csv` for `4_2_evaluate_test_predictions.ipynb`.

**Interpretation:** The saved component models were trained through 2024-07-01 and the residual stackers through 2025-01-01. Predictions for 2022–2024 therefore include training/validation overlap and are not out-of-sample backtest results. The 2025 predictions are out of sample. If interrupted, rerunning resumes from the last completed month in the temporary CSV. A true out-of-sample backtest for the earlier years requires models retrained with chronological cutoffs.


In [1]:
# Imports and final model

import gc
import json
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.exceptions import InconsistentVersionWarning
from tqdm import tqdm

sys.path.append(os.path.abspath("../../"))
from EPF import variables
sys.path.append(os.path.abspath("../2_Features_build"))
import target_features
import value_signal
AEMO_MAX_HORIZON = 78
HORIZON_LIST = range(1, variables.HORIZON_COUNT + 1)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", InconsistentVersionWarning)
    final_params = joblib.load(variables.FINAL_PARAMS_PATH)
    value_signal_model = value_signal.ValueSignalModel.load(variables.VALUE_SIGNAL_MODEL_PATH)
meta_feature_names = final_params["stackers"][0].feature_name_
BACKTEST_START = pd.Timestamp("2022-01-01")
BACKTEST_END_EXCLUSIVE = pd.Timestamp("2026-01-01")
BACKTEST_ORIGIN_CADENCE = "24h"


In [2]:
# Load selected features and data schemas

objectives = ("normal", "arcsinh", "spikes", "dips")
selected_by_objective = {
    objective: pd.read_parquet(variables.SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet")
    for objective in objectives
}
all_selected_features = sorted({
    feature
    for selected in selected_by_objective.values()
    for horizon in HORIZON_LIST
    for feature in selected.loc[selected[f"h{horizon}"].astype(bool), "feature"]
})

target_columns = [f"target_h{horizon}" for horizon in HORIZON_LIST]
target_index_col = pq.ParquetFile(variables.AGG_TARGET_DATASET_PATH).schema_arrow.pandas_metadata["index_columns"][0]

feature_schema = pq.ParquetFile(variables.FEATURES_DATASET_PATH).schema_arrow
missing_features = sorted(set(all_selected_features) - set(feature_schema.names))
if missing_features:
    raise ValueError(
        f"{variables.FEATURES_DATASET_PATH} is missing {len(missing_features)} selected feature(s): "
        f"{missing_features}. Rebuild the affected source feature notebook, then 99_all.ipynb."
    )
feature_index_col = feature_schema.pandas_metadata["index_columns"][0]


In [3]:
# Load heuristic-archive and AEMO benchmark schemas

observations_per_target = variables.FEATURE_STEPS_PER_HORIZON
heuristic_archives = {
    "naive": variables.NAIVE_FORECASTS_PATH,
    "operating_protocol": variables.OPERATING_PROTOCOL_FORECASTS_PATH,
}
heuristic_columns = [f"predicted_h{horizon}" for horizon in HORIZON_LIST]
missing_archives = [str(path) for path in heuristic_archives.values() if not path.exists()]
if missing_archives:
    raise FileNotFoundError(f"Run both Heuristic notebooks first. Missing: {missing_archives}")
heuristic_index_cols = {
    name: pq.ParquetFile(path).schema_arrow.pandas_metadata["index_columns"][0]
    for name, path in heuristic_archives.items()
}

aemo_schema = pq.ParquetFile(variables.AEMO_PREDISPATCH_PATH).schema_arrow
aemo_index_col = aemo_schema.pandas_metadata["index_columns"][0]
available_aemo_columns = set(aemo_schema.names) - {aemo_index_col}
aemo_columns = [
    f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon}"
    for horizon in range(1, AEMO_MAX_HORIZON + 1)
    if f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon}" in available_aemo_columns
]

# Load each component model once, then reuse it across every backtest month.
# Loading inside the monthly loop would deserialize all 2+ GB of models 48 times.
MODEL_FILENAMES = {
    "base_l1": "full_range_regressor_clipped_MAE_loss",
    "base_l2": "full_range_regressor_unclipped_RMSE_loss",
    "spike_probability": "positive_spike_classifier_binary_loss",
    "spike_mae": "positive_spike_regressor_unclipped_mae_loss",
    "spike_q90": "positive_spike_regressor_unclipped_quantile_loss",
    "dip_probability": "negative_spike_classifer_unclipped_binary_loss",
    "dip_mae": "negative_spike_regressor_unclipped_mae_loss",
    "dip_q10": "negative_spike_regressor_unclipped_quantile_loss",
}
models_by_horizon = {horizon: {} for horizon in HORIZON_LIST}
with warnings.catch_warnings():
    # These models were trained with sklearn 1.9.0 and are being read by 1.9.1.
    # Suppress the repeated warning here only; it otherwise hides useful progress.
    warnings.simplefilter("ignore", InconsistentVersionWarning)
    model_bar = tqdm(total=len(HORIZON_LIST) * len(MODEL_FILENAMES), desc="Loading component models", unit="model")
    for horizon in HORIZON_LIST:
        for output_name, model_name in MODEL_FILENAMES.items():
            model_path = variables.TRAINED_MODELS_PATH / f"h{horizon:02d}_{model_name}.joblib"
            models_by_horizon[horizon][output_name] = joblib.load(model_path)
            model_bar.update(1)
    model_bar.close()


Loading component models: 100%|██████████| 768/768 [00:49<00:00, 15.65model/s] 


In [4]:
# Generate and append one month at a time. A checkpoint allows safe resumption.

variables.MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
temporary_output = variables.ACTUAL_VS_PREDICTED_TEST_SET.with_suffix(".csv.tmp")
checkpoint_path = variables.ACTUAL_VS_PREDICTED_TEST_SET.with_suffix(".csv.checkpoint")
checkpoint_next = variables.ACTUAL_VS_PREDICTED_TEST_SET.with_suffix(".csv.checkpoint.tmp")
expected_header = "forecast_origin," + ",".join(
    f"{name}_h{horizon}"
    for name in ("actual", "predicted", "value_signal", "naive", "operating_protocol", "aemo_predispatch")
    for horizon in HORIZON_LIST
)
month_starts = pd.date_range(BACKTEST_START, BACKTEST_END_EXCLUSIVE, freq="MS", inclusive="left")
rows_written = 0
resume_from = BACKTEST_START

if temporary_output.exists():
    with temporary_output.open("rb") as output_file:
        header = output_file.readline().decode().rstrip("\r\n")
        if header != expected_header:
            raise ValueError(f"Unexpected columns in {temporary_output}; cannot resume safely.")
        if checkpoint_path.exists():
            checkpoint = json.loads(checkpoint_path.read_text())
            saved_size = checkpoint["file_size"]
            if temporary_output.stat().st_size < saved_size:
                raise ValueError("Temporary prediction CSV is shorter than its checkpoint.")
            rows_written = checkpoint["rows_written"]
            resume_from = pd.Timestamp(checkpoint["last_completed_month"]) + pd.DateOffset(months=1)
        else:
            # Recover the completed 2022–2024 output from the earlier yearly version.
            output_file.seek(0, os.SEEK_END)
            output_file.seek(max(0, output_file.tell() - 65536))
            last_row = output_file.read().splitlines()[-1]
            if len(last_row.split(b",")) != len(expected_header.split(",")):
                raise ValueError("Temporary prediction CSV has an incomplete final row.")
            last_date = pd.Timestamp(last_row.split(b",", 1)[0].decode())
            next_month = last_date.to_period("M").to_timestamp() + pd.DateOffset(months=1)
            if last_date != next_month - pd.Timedelta(minutes=variables.FEATURE_GRANULARITY_IN_MINUTES):
                raise ValueError("Temporary prediction CSV ends mid-month and has no checkpoint.")
            resume_from = next_month
            output_file.seek(0)
            rows_written = sum(1 for _ in output_file) - 1
            checkpoint = {
                "last_completed_month": last_date.strftime("%Y-%m"),
                "rows_written": rows_written,
                "file_size": temporary_output.stat().st_size,
            }
            checkpoint_next.write_text(json.dumps(checkpoint))
            os.replace(checkpoint_next, checkpoint_path)
    if checkpoint_path.exists():
        with temporary_output.open("r+b") as output_file:
            output_file.truncate(checkpoint["file_size"])
    print(f"Resuming at {resume_from:%Y-%m}; {rows_written:,} rows already saved.")
else:
    checkpoint_path.unlink(missing_ok=True)

remaining_months = month_starts[month_starts >= resume_from]
for period_start in tqdm(remaining_months, desc="Backtesting months", unit="month"):
    period_end = min(BACKTEST_END_EXCLUSIVE, period_start + pd.DateOffset(months=1))
    target_filters = [(target_index_col, ">=", period_start), (target_index_col, "<", period_end)]
    targets_test = pd.read_parquet(
        variables.AGG_TARGET_DATASET_PATH, columns=target_columns, filters=target_filters
    ).sort_index().astype(np.float32).dropna(subset=target_columns)
    test_index = targets_test.index[targets_test.index.floor(BACKTEST_ORIGIN_CADENCE) == targets_test.index]
    targets_test = targets_test.loc[test_index]
    if test_index.empty:
        raise ValueError(f"No complete target rows found for {period_start:%Y-%m}.")

    features_test = pd.read_parquet(
        variables.FEATURES_DATASET_PATH,
        columns=all_selected_features,
        filters=[(feature_index_col, ">=", period_start), (feature_index_col, "<", period_end)],
    ).sort_index().astype(np.float32).reindex(test_index)
    aemo_test = pd.read_parquet(
        variables.AEMO_PREDISPATCH_PATH,
        columns=aemo_columns,
        filters=[(aemo_index_col, ">=", period_start), (aemo_index_col, "<", period_end)],
    ).sort_index().astype(np.float32).reindex(test_index)
    heuristic_test = {
        name: pd.read_parquet(
            path, columns=heuristic_columns,
            filters=[(heuristic_index_cols[name], ">=", period_start), (heuristic_index_cols[name], "<", period_end)],
        ).sort_index().astype(np.float32).reindex(test_index)
        for name, path in heuristic_archives.items()
    }
    aemo_offset = ((test_index.minute % 30) // variables.FEATURE_GRANULARITY_IN_MINUTES).astype(int)
    aemo_aligned = aemo_offset == 0
    aemo_weight = aemo_offset.astype(np.float32) / observations_per_target

    shape = (len(test_index), len(HORIZON_LIST))
    predictions = {
        "actual": targets_test[target_columns].values.astype(np.float32),
        "predicted": np.full(shape, np.nan, dtype=np.float32),
        "naive": heuristic_test["naive"][heuristic_columns].values,
        "operating_protocol": heuristic_test["operating_protocol"][heuristic_columns].values,
        "aemo_predispatch": np.full(shape, np.nan, dtype=np.float32),
    }
    component_predictions = {
        name: np.full(shape, np.nan, dtype=np.float32)
        for name in value_signal.COMPONENT_NAMES
    }

    for column, horizon in enumerate(tqdm(HORIZON_LIST, desc=f"{period_start:%Y-%m} horizons", leave=False)):
        selected = selected_by_objective["normal"]
        feature_names = selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()
        model_input = target_features.append_target_time_feats(features_test[feature_names], horizon)
        model = models_by_horizon[horizon]["base_l1"]
        base_l1 = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        del model_input, model

        selected = selected_by_objective["arcsinh"]
        feature_names = selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()
        model_input = target_features.append_target_time_feats(features_test[feature_names], horizon)
        model = models_by_horizon[horizon]["base_l2"]
        base_l2 = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        del model_input, model

        selected = selected_by_objective["spikes"]
        feature_names = selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()
        model_input = target_features.append_target_time_feats(features_test[feature_names], horizon)
        model = models_by_horizon[horizon]["spike_probability"]
        spike_probability = np.asarray(model.predict_proba(model_input)[:, 1], dtype=np.float32)
        model = models_by_horizon[horizon]["spike_mae"]
        spike_mae = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        model = models_by_horizon[horizon]["spike_q90"]
        spike_q90 = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        del model_input, model

        selected = selected_by_objective["dips"]
        feature_names = selected.loc[selected[f"h{horizon}"].astype(bool), "feature"].tolist()
        model_input = target_features.append_target_time_feats(features_test[feature_names], horizon)
        model = models_by_horizon[horizon]["dip_probability"]
        dip_probability = np.asarray(model.predict_proba(model_input)[:, 1], dtype=np.float32)
        model = models_by_horizon[horizon]["dip_mae"]
        dip_mae = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        model = models_by_horizon[horizon]["dip_q10"]
        dip_q10 = np.asarray(np.sinh(model.predict(model_input)) * variables.PRICE_TRANSFORM_SCALE, dtype=np.float32)
        del model_input, model

        spike_probability = np.clip(spike_probability, 1e-6, 1 - 1e-6)
        dip_probability = np.clip(dip_probability, 1e-6, 1 - 1e-6)
        spike_mae_gap = np.maximum(spike_mae - base_l1, 0)
        spike_q90_gap = np.maximum(spike_q90 - base_l1, 0)
        dip_mae_gap = np.minimum(dip_mae - base_l1, 0)
        dip_q10_gap = np.minimum(dip_q10 - base_l1, 0)
        meta_values = {
            "base_l1": base_l1,
            "base_l2": base_l2,
            "spike_mae": spike_mae,
            "spike_q90": spike_q90,
            "dip_mae": dip_mae,
            "dip_q10": dip_q10,
            "spike_probability": spike_probability,
            "dip_probability": dip_probability,
            "spike_logit": np.log(spike_probability / (1 - spike_probability)),
            "dip_logit": np.log(dip_probability / (1 - dip_probability)),
            "base_l2_minus_l1": base_l2 - base_l1,
            "spike_mae_positive_gap": spike_mae_gap,
            "spike_q90_positive_gap": spike_q90_gap,
            "dip_mae_negative_gap": dip_mae_gap,
            "dip_q10_negative_gap": dip_q10_gap,
            "weighted_spike_mae_gap": spike_probability * spike_mae_gap,
            "weighted_spike_q90_gap": spike_probability * spike_q90_gap,
            "weighted_dip_mae_gap": dip_probability * dip_mae_gap,
            "weighted_dip_q10_gap": dip_probability * dip_q10_gap,
        }
        meta_features = pd.DataFrame(
            np.column_stack([meta_values[name] for name in meta_feature_names]).astype(np.float32),
            columns=meta_feature_names,
        )
        predictions["predicted"][:, column] = base_l1 + final_params["stackers"][horizon - 1].predict(meta_features)
        for name, values in (
            ("base_l1", base_l1), ("base_l2", base_l2),
            ("spike_probability", spike_probability), ("spike_mae", spike_mae),
            ("spike_q90", spike_q90), ("dip_probability", dip_probability),
            ("dip_mae", dip_mae), ("dip_q10", dip_q10),
        ):
            component_predictions[name][:, column] = values

        current_col = f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon}"
        next_col = f"predispatch_rrp_{variables.TARGET_REGION}_h{horizon + 1}"
        if horizon <= AEMO_MAX_HORIZON and current_col in aemo_test:
            aemo_prediction = predictions["aemo_predispatch"][:, column]
            current_price = aemo_test[current_col].values.astype(np.float32)
            aemo_prediction[aemo_aligned] = current_price[aemo_aligned]
            if (~aemo_aligned).any() and next_col in aemo_test:
                next_price = aemo_test[next_col].values.astype(np.float32)
                aemo_prediction[~aemo_aligned] = (
                    (1 - aemo_weight[~aemo_aligned]) * current_price[~aemo_aligned]
                    + aemo_weight[~aemo_aligned] * next_price[~aemo_aligned]
                )
        del meta_values, meta_features
    predictions["value_signal"] = value_signal_model.predict(
        component_predictions, predictions["predicted"], test_index
    )
    prediction_tables = [
        pd.DataFrame(matrix, index=test_index, columns=[f"{name}_h{horizon}" for horizon in HORIZON_LIST])
        for name, matrix in predictions.items()
    ]
    all_predictions = pd.concat(prediction_tables, axis=1)
    all_predictions.index.name = "forecast_origin"
    all_predictions.to_csv(
        temporary_output,
        mode="w" if rows_written == 0 else "a",
        header=rows_written == 0,
    )
    rows_written += len(all_predictions)
    checkpoint = {
        "last_completed_month": period_start.strftime("%Y-%m"),
        "rows_written": rows_written,
        "file_size": temporary_output.stat().st_size,
    }
    checkpoint_next.write_text(json.dumps(checkpoint))
    os.replace(checkpoint_next, checkpoint_path)
    print(f"Completed {period_start:%Y-%m}: {len(all_predictions):,} rows")
    del targets_test, features_test, aemo_test, heuristic_test, predictions, component_predictions, prediction_tables, all_predictions
    gc.collect()

os.replace(temporary_output, variables.ACTUAL_VS_PREDICTED_TEST_SET)
checkpoint_path.unlink(missing_ok=True)


Backtesting months:   2%|▏         | 1/48 [00:52<40:44, 52.01s/month]

Completed 2022-01: 8,928 rows


Backtesting months:   4%|▍         | 2/48 [01:42<39:16, 51.22s/month]

Completed 2022-02: 8,064 rows


Backtesting months:   6%|▋         | 3/48 [02:36<39:25, 52.57s/month]

...

Completed 2025-10: 8,928 rows


Backtesting months:  98%|█████████▊| 47/48 [41:12<00:50, 50.99s/month]

Completed 2025-11: 8,640 rows


Backtesting months: 100%|██████████| 48/48 [42:03<00:00, 52.58s/month]

Completed 2025-12: 8,928 rows


In [5]:
print(f"Saved {rows_written:,} prediction rows (2022–2025) to {variables.ACTUAL_VS_PREDICTED_TEST_SET}")


Saved 420,768 prediction rows (2022–2025) to C:\Users\Daniel\OneDrive\Documents\2_Energy_toolkit\1_ST_price_prediction\EPF\5_Model\Data\5_model_results\actual_vs_predicted_test_set.csv
